In [1]:
import json
import numpy as np
from pandas import read_csv
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, KFold, ParameterGrid
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
import joblib


In [2]:
#Variables generales
ruta_model = "../../Model/"
ruta_metrics = "../../Metrics/"
ruta_df = "../../Datasets/"

semilla = 111

## FUNCIÓN BASE

In [3]:
def train_logistic_regression(X, y, output_model_path = 'best_model.pkl', output_metrics_path = 'metrics.json', kf = 2, param_grid = {'C' : [0.1]}):  
    
    # Initialize Logistic Regression
    lr = LogisticRegression(random_state = 42)
    
    # Definir validación cruzada de K-Fold
    kf = KFold(n_splits = kf
               , shuffle = True
               , random_state = 42
               )
    
    # Cross Validation
    grid_search = GridSearchCV(estimator = lr
                               , param_grid = param_grid
                               , cv = kf
                               , scoring = 'accuracy'
                               , n_jobs = -1
                               , verbose = 2
                               )
    
    # Ajustar el modelo
    grid_search.fit(X, y)
    
    # mejor modelo
    best_model = grid_search.best_estimator_
    
    # Guarde el mejor modelo en un archivo .pkl
    joblib.dump(best_model, output_model_path)
    
    # mejor modelo.
    y_pred = best_model.predict(X)
    
    # Calcular métricas de evaluación
    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average = 'weighted'),
        'recall': recall_score(y, y_pred, average = 'weighted'),
        'f1_score': f1_score(y, y_pred, average = 'weighted'),
        'roc_auc': roc_auc_score(y, y_pred, multi_class = 'ovr'),
        'confusion_matrix': confusion_matrix(y, y_pred).tolist(),
        'classification_report': classification_report(y, y_pred, output_dict = True),
        'balanced_accuracy': balanced_accuracy_score(y, y_pred),
        'cohen_kappa': cohen_kappa_score(y, y_pred),
        'matthews_corrcoef': matthews_corrcoef(y, y_pred)
    }
    
    # Guarde las métricas en un archivo JSON
    with open(output_metrics_path, 'w') as f:
        json.dump(metrics, f, indent=4)
        
    return metrics, best_model

## Entrenamiento

### df interpolation

* SMOTE

In [4]:
# carga de caracteristicas
df_IM_smote = read_csv('{}Interpolation_Method_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [5]:
%%time
param_grid = [
    {'solver': ['lbfgs']
     , 'max_iter': [2000, 3000, 5000]
     , 'penalty': ['l2', None]
     , 'C': [0.1, 1, 10, 100]
    }
]

metric_1, model_1 = train_logistic_regression(X = df_IM_smote.drop(columns='FLAG')
                                              , y = df_IM_smote['FLAG']
                                              , output_model_path = '{}RL_IM_S.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}RL_IM_S.json'.format(ruta_metrics)
                                              , kf = 3
                                              , param_grid = param_grid)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py:378: FitFailedWarning: 
36 fits failed out of a total of 72.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
36 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_logistic.py", line 1091, in fit
    solver = _check_solver(self.solver, self.penalty, self.dual)
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\linea

Wall time: 5.39 s


In [6]:
metric_1

{'accuracy': 0.5872759600253418,
 'precision': 0.6603305322781187,
 'recall': 0.5872759600253418,
 'f1_score': 0.5646568858404044,
 'roc_auc': 0.6150118848633652,
 'confusion_matrix': [[9933, 17251], [2944, 18803]],
 'classification_report': {'0': {'precision': 0.7713753203385881,
   'recall': 0.36539876397881105,
   'f1-score': 0.4958937620129303,
   'support': 27184},
  '1': {'precision': 0.5215232706495811,
   'recall': 0.8646250057479192,
   'f1-score': 0.6506115811145136,
   'support': 21747},
  'accuracy': 0.5872759600253418,
  'macro avg': {'precision': 0.6464492954940846,
   'recall': 0.6150118848633651,
   'f1-score': 0.573252671563722,
   'support': 48931},
  'weighted avg': {'precision': 0.6603305322781187,
   'recall': 0.5872759600253418,
   'f1-score': 0.5646568858404044,
   'support': 48931}},
 'balanced_accuracy': 0.6150118848633651,
 'cohen_kappa': 0.21582450205824522,
 'matthews_corrcoef': 0.2595643235245292}

* ADASYN

In [7]:
# carga de caracteristicas
df_IM_adasyn = read_csv('{}Interpolation_Method_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [8]:
%%time
param_grid = [
    {'solver': ['lbfgs']
     , 'max_iter': [2000, 3000, 5000]
     , 'penalty': ['l2']
     , 'C': [0.1, 1, 10, 100]
    }
]

metric_2, model_2 = train_logistic_regression(X = df_IM_adasyn.drop(columns='FLAG')
                                              , y = df_IM_adasyn['FLAG']
                                              , output_model_path = '{}RL_IM_A.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}RL_IM_A.json'.format(ruta_metrics)
                                              , kf = 3
                                              , param_grid = param_grid)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Wall time: 2.02 s


In [9]:
metric_2

{'accuracy': 0.6089723910685414,
 'precision': 0.6369088336601445,
 'recall': 0.6089723910685414,
 'f1_score': 0.5491248392064051,
 'roc_auc': 0.569264789403784,
 'confusion_matrix': [[25052, 2132], [16974, 4703]],
 'classification_report': {'0': {'precision': 0.596107171750821,
   'recall': 0.9215715126545027,
   'f1-score': 0.7239416269325243,
   'support': 27184},
  '1': {'precision': 0.6880760790051207,
   'recall': 0.21695806615306545,
   'f1-score': 0.32989618406285076,
   'support': 21677},
  'accuracy': 0.6089723910685414,
  'macro avg': {'precision': 0.6420916253779708,
   'recall': 0.5692647894037841,
   'f1-score': 0.5269189054976875,
   'support': 48861},
  'weighted avg': {'precision': 0.6369088336601445,
   'recall': 0.6089723910685414,
   'f1-score': 0.5491248392064051,
   'support': 48861}},
 'balanced_accuracy': 0.5692647894037841,
 'cohen_kappa': 0.14885297226256455,
 'matthews_corrcoef': 0.19841317000488176}

### df linear regression

* SMOTE

In [10]:
# carga de caracteristicas
df_LR_smote = read_csv('{}Linear_Regression_SMOTE_Train_Modificado.csv'.format(ruta_df))

In [11]:
%%time
param_grid = [
    {'solver': ['lbfgs']
     , 'max_iter': [2000, 3000, 5000]
     , 'penalty': ['l2', None]
     , 'C': [0.1, 1, 10, 100]
    }
]

metric_3, model_3 = train_logistic_regression(X = df_LR_smote.drop(columns='FLAG')
                                              , y = df_LR_smote['FLAG']
                                              , output_model_path = '{}RL_LR_S.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}RL_LR_S.json'.format(ruta_metrics)
                                              , kf = 3
                                              , param_grid = param_grid)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py:378: FitFailedWarning: 
36 fits failed out of a total of 72.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
36 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_logistic.py", line 1091, in fit
    solver = _check_solver(self.solver, self.penalty, self.dual)
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\linea

Wall time: 2.73 s


In [12]:
metric_3

{'accuracy': 0.5496719870838528,
 'precision': 0.6165756763316504,
 'recall': 0.5496719870838528,
 'f1_score': 0.5211395335498236,
 'roc_auc': 0.5787496341495068,
 'confusion_matrix': [[8619, 18565], [3470, 18277]],
 'classification_report': {'0': {'precision': 0.7129621970386302,
   'recall': 0.31706150676868744,
   'f1-score': 0.4389275074478649,
   'support': 27184},
  '1': {'precision': 0.4960914174040497,
   'recall': 0.840437761530326,
   'f1-score': 0.6239055112734472,
   'support': 21747},
  'accuracy': 0.5496719870838528,
  'macro avg': {'precision': 0.6045268072213399,
   'recall': 0.5787496341495068,
   'f1-score': 0.531416509360656,
   'support': 48931},
  'weighted avg': {'precision': 0.6165756763316504,
   'recall': 0.5496719870838528,
   'f1-score': 0.5211395335498236,
   'support': 48931}},
 'balanced_accuracy': 0.5787496341495068,
 'cohen_kappa': 0.14727617696564987,
 'matthews_corrcoef': 0.18145465359143081}

* ADASYN

In [13]:
# carga de caracteristicas
df_LR_adasyn = read_csv('{}Linear_Regression_ADASYN_Train_Modificado.csv'.format(ruta_df))

In [14]:
%%time
param_grid = [
    {'solver': ['lbfgs']
     , 'max_iter': [2000, 3000, 5000]
     , 'penalty': ['l2', None]
     , 'C': [0.1, 1, 10, 100]
    }
]

metric_4, model_4 = train_logistic_regression(X = df_LR_adasyn.drop(columns='FLAG')
                                              , y = df_LR_adasyn['FLAG']
                                              , output_model_path = '{}RL_LR_A.pkl'.format(ruta_model)
                                              , output_metrics_path = '{}RL_LR_A.json'.format(ruta_metrics)
                                              , kf = 3
                                              , param_grid = param_grid)

Fitting 3 folds for each of 24 candidates, totalling 72 fits


C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py:378: FitFailedWarning: 
36 fits failed out of a total of 72.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
36 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\model_selection\_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_logistic.py", line 1091, in fit
    solver = _check_solver(self.solver, self.penalty, self.dual)
  File "C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\linea

Wall time: 1.37 s


C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\pedro\AppData\Roaming\Python\Python310\site-packages\sklearn\metrics\_classification.py:1334: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\pe

In [15]:
metric_4

{'accuracy': 0.4416350005135052,
 'precision': 0.19504147367856375,
 'recall': 0.4416350005135052,
 'f1_score': 0.2705837103137627,
 'roc_auc': 0.5,
 'confusion_matrix': [[0, 27184], [0, 21501]],
 'classification_report': {'0': {'precision': 0.0,
   'recall': 0.0,
   'f1-score': 0.0,
   'support': 27184},
  '1': {'precision': 0.4416350005135052,
   'recall': 1.0,
   'f1-score': 0.6126862907132476,
   'support': 21501},
  'accuracy': 0.4416350005135052,
  'macro avg': {'precision': 0.2208175002567526,
   'recall': 0.5,
   'f1-score': 0.3063431453566238,
   'support': 48685},
  'weighted avg': {'precision': 0.19504147367856375,
   'recall': 0.4416350005135052,
   'f1-score': 0.2705837103137627,
   'support': 48685}},
 'balanced_accuracy': 0.5,
 'cohen_kappa': 0.0,
 'matthews_corrcoef': 0.0}